# PARSER CODE FOR PROJECT (Kolesa.kz вторичный и первичный)

In [12]:
!pip install selenium webdriver-manager pandas --quiet

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time
import re

options = webdriver.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

BASE_URL = "https://kolesa.kz/cars/avtomobili-s-probegom/almaty/"

all_rows = []
MAX_PAGES = 250


def clean_price(text):
    return int(re.sub(r'\D', '', text)) if text else None


def parse_title(title):
    brand = model = year = None

    if not title:
        return brand, model, year

    parts = title.split()
    if len(parts) >= 2:
        brand = parts[0]
        model = parts[1]

    m = re.search(r'(19\d{2}|20\d{2})', title)
    if m:
        year = int(m.group(1))

    return brand, model, year


for page in range(1, MAX_PAGES + 1):
    url = BASE_URL if page == 1 else f"{BASE_URL}?page={page}"

    print("Page:", page)
    driver.get(url)

    # ЖДЕМ загрузку
    time.sleep(3)

    # СКРОЛЛ (ключевой фикс)
    for _ in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    cards = driver.find_elements(By.CLASS_NAME, "a-card")

    print("Cards found:", len(cards))

    if not cards:
        break

    for card in cards:
        try:
            title = card.find_element(By.CLASS_NAME, "a-card__title").text
        except:
            title = None

        try:
            price = clean_price(card.find_element(By.CLASS_NAME, "a-card__price").text)
        except:
            price = None

        try:
            link = card.find_element(By.TAG_NAME, "a").get_attribute("href")
        except:
            link = None

        brand, model, year = parse_title(title)

        try:
            info = card.find_element(By.CLASS_NAME, "a-card__subtitle").text.lower()
        except:
            info = ""

        # --- пробег ---
        mileage = None
        m = re.search(r'([\d\s]+)\s*км', info)
        if m:
            mileage = int(m.group(1).replace(" ", ""))

        # --- двигатель ---
        engine = None
        m = re.search(r'([\d.]+)\s*l', info)
        if m:
            engine = float(m.group(1))

        # --- коробка ---
        transmission = None
        if "автомат" in info:
            transmission = "automatic"
        elif "механика" in info:
            transmission = "manual"

        all_rows.append({
            "price": price,
            "brand": brand,
            "model": model,
            "year": year,
            "mileage_km": mileage,
            "engine_l": engine,
            "transmission": transmission,
            "title": title,
            "link": link,
            "page": page
        })

driver.quit()

df_cars = pd.DataFrame(all_rows)
df_cars.to_csv("kolesa_secondary_almaty_FIXED.csv", index=False)

print("DONE:", len(df_cars))
df_cars.info()
df_cars.head()

Page: 1
Cards found: 20
Page: 2
Cards found: 20
Page: 3
Cards found: 20
Page: 4
Cards found: 20
Page: 5
Cards found: 20
Page: 6
Cards found: 20
Page: 7
Cards found: 20
Page: 8
Cards found: 20
Page: 9
Cards found: 20
Page: 10
Cards found: 20
Page: 11
Cards found: 20
Page: 12
Cards found: 20
Page: 13
Cards found: 20
Page: 14
Cards found: 20
Page: 15
Cards found: 20
Page: 16
Cards found: 20
Page: 17
Cards found: 20
Page: 18
Cards found: 20
Page: 19
Cards found: 20
Page: 20
Cards found: 20
Page: 21
Cards found: 20
Page: 22
Cards found: 20
Page: 23
Cards found: 20
Page: 24
Cards found: 20
Page: 25
Cards found: 20
Page: 26
Cards found: 20
Page: 27
Cards found: 20
Page: 28
Cards found: 20
Page: 29
Cards found: 20
Page: 30
Cards found: 20
Page: 31
Cards found: 20
Page: 32
Cards found: 20
Page: 33
Cards found: 20
Page: 34
Cards found: 20
Page: 35
Cards found: 20
Page: 36
Cards found: 20
Page: 37
Cards found: 20
Page: 38
Cards found: 20
Page: 39
Cards found: 20
Page: 40
Cards found: 20
Page: 41


,price,brand,model,year,mileage_km,engine_l,transmission,title,link,page
0,20000000,Hyundai,Santa,None,None,None,None,Hyundai Santa Fe,https://kolesa.kz/a/show/211674976?search_id=9...,1
1,7999000,Hyundai,Grandeur,None,None,None,None,Hyundai Grandeur,https://kolesa.kz/a/show/212895246?search_id=9...,1
2,29770000,Audi,Q7,None,None,None,None,Audi Q7,https://kolesa.kz/a/show/194067401?search_id=9...,1
3,39999999,Mercedes-Benz,E,None,None,None,None,Mercedes-Benz E 53 AMG,https://kolesa.kz/a/show/188597624?search_id=9...,1
4,7000000,Lexus,RX,None,None,None,None,Lexus RX 330,https://kolesa.kz/a/show/218781975?search_id=9...,1


In [19]:
print(df_cars.head())

      price          brand     model  year mileage_km engine_l transmission  \
0  20000000        Hyundai     Santa  None       None     None         None   
1   7999000        Hyundai  Grandeur  None       None     None         None   
2  29770000           Audi        Q7  None       None     None         None   
3  39999999  Mercedes-Benz         E  None       None     None         None   
4   7000000          Lexus        RX  None       None     None         None   

                    title                                               link  \
0        Hyundai Santa Fe  https://kolesa.kz/a/show/211674976?search_id=9...   
1        Hyundai Grandeur  https://kolesa.kz/a/show/212895246?search_id=9...   
2                 Audi Q7  https://kolesa.kz/a/show/194067401?search_id=9...   
3  Mercedes-Benz E 53 AMG  https://kolesa.kz/a/show/188597624?search_id=9...   
4            Lexus RX 330  https://kolesa.kz/a/show/218781975?search_id=9...   

   page  
0     1  
1     1  
2     1  
3   

In [21]:
df_cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   price         5000 non-null   int64 
 1   brand         5000 non-null   object
 2   model         5000 non-null   object
 3   year          0 non-null      object
 4   mileage_km    0 non-null      object
 5   engine_l      0 non-null      object
 6   transmission  0 non-null      object
 7   title         5000 non-null   object
 8   link          5000 non-null   object
 9   page          5000 non-null   int64 
dtypes: int64(2), object(8)
memory usage: 390.8+ KB


In [23]:
!pip install selenium webdriver-manager pandas --quiet

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time
import re

options = webdriver.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

BASE_URL = "https://kolesa.kz/cars/novye-avtomobili/almaty/"

all_rows = []
MAX_PAGES = 250


def clean_price(text):
    return int(re.sub(r'\D', '', text)) if text else None


def parse_title(title):
    brand = model = year = None

    if not title:
        return brand, model, year

    parts = title.split()
    if len(parts) >= 2:
        brand = parts[0]
        model = parts[1]

    m = re.search(r'(19\d{2}|20\d{2})', title)
    if m:
        year = int(m.group(1))

    return brand, model, year


for page in range(1, MAX_PAGES + 1):
    url = BASE_URL if page == 1 else f"{BASE_URL}?page={page}"

    print("Page:", page)
    driver.get(url)

    # ЖДЕМ загрузку
    time.sleep(3)

    # СКРОЛЛ (ключевой фикс)
    for _ in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    cards = driver.find_elements(By.CLASS_NAME, "a-card")

    print("Cards found:", len(cards))

    if not cards:
        break

    for card in cards:
        try:
            title = card.find_element(By.CLASS_NAME, "a-card__title").text
        except:
            title = None

        try:
            price = clean_price(card.find_element(By.CLASS_NAME, "a-card__price").text)
        except:
            price = None

        try:
            link = card.find_element(By.TAG_NAME, "a").get_attribute("href")
        except:
            link = None

        brand, model, year = parse_title(title)

        try:
            info = card.find_element(By.CLASS_NAME, "a-card__subtitle").text.lower()
        except:
            info = ""

        # --- пробег ---
        mileage = None
        m = re.search(r'([\d\s]+)\s*км', info)
        if m:
            mileage = int(m.group(1).replace(" ", ""))

        # --- двигатель ---
        engine = None
        m = re.search(r'([\d.]+)\s*l', info)
        if m:
            engine = float(m.group(1))

        # --- коробка ---
        transmission = None
        if "автомат" in info:
            transmission = "automatic"
        elif "механика" in info:
            transmission = "manual"

        all_rows.append({
            "price": price,
            "brand": brand,
            "model": model,
            "year": year,
            "mileage_km": mileage,
            "engine_l": engine,
            "transmission": transmission,
            "title": title,
            "link": link,
            "page": page
        })

driver.quit()

df_primary_cars = pd.DataFrame(all_rows)
df_primary_cars.to_csv("kolesa_primary_almaty_FIXED.csv", index=False)

print("DONE:", len(df_primary_cars))
df_primary_cars.info()
df_primary_cars.head()

Page: 1
Cards found: 20
Page: 2
Cards found: 20
Page: 3
Cards found: 20
Page: 4
Cards found: 20
Page: 5
Cards found: 20
Page: 6
Cards found: 20
Page: 7
Cards found: 20
Page: 8
Cards found: 20
Page: 9
Cards found: 20
Page: 10
Cards found: 20
Page: 11
Cards found: 20
Page: 12
Cards found: 20
Page: 13
Cards found: 20
Page: 14
Cards found: 20
Page: 15
Cards found: 20
Page: 16
Cards found: 20
Page: 17
Cards found: 20
Page: 18
Cards found: 20
Page: 19
Cards found: 20
Page: 20
Cards found: 20
Page: 21
Cards found: 20
Page: 22
Cards found: 20
Page: 23
Cards found: 20
Page: 24
Cards found: 20
Page: 25
Cards found: 20
Page: 26
Cards found: 20
Page: 27
Cards found: 20
Page: 28
Cards found: 20
Page: 29
Cards found: 20
Page: 30
Cards found: 20
Page: 31
Cards found: 20
Page: 32
Cards found: 20
Page: 33
Cards found: 20
Page: 34
Cards found: 20
Page: 35
Cards found: 20
Page: 36
Cards found: 20
Page: 37
Cards found: 20
Page: 38
Cards found: 20
Page: 39
Cards found: 20
Page: 40
Cards found: 20
Page: 41


,price,brand,model,year,mileage_km,engine_l,transmission,title,link,page
0,52500000,Zeekr,8x,None,None,None,None,Zeekr 8x Shadow,https://kolesa.kz/a/show/218477247?search_id=4...,1
1,48800000,BMW,X5,None,None,None,None,BMW X5 M,https://kolesa.kz/a/show/218615175?search_id=4...,1
2,15000000,Hyundai,Avante,None,None,None,None,Hyundai Avante,https://kolesa.kz/a/show/218502983?search_id=4...,1
3,12499000,BYD,Song,None,None,None,None,BYD Song Plus,https://kolesa.kz/a/show/212233412?search_id=4...,1
4,6600000,Changan,Qiyuan,None,None,None,None,Changan Qiyuan Q05,https://kolesa.kz/a/show/218269118?search_id=4...,1


In [24]:
df_primary_cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2552 entries, 0 to 2551
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   price         2552 non-null   int64 
 1   brand         2552 non-null   object
 2   model         2552 non-null   object
 3   year          0 non-null      object
 4   mileage_km    0 non-null      object
 5   engine_l      0 non-null      object
 6   transmission  0 non-null      object
 7   title         2552 non-null   object
 8   link          2552 non-null   object
 9   page          2552 non-null   int64 
dtypes: int64(2), object(8)
memory usage: 199.5+ KB
